# MENTORX: the deployed DKT-Q vs LSTM variations

**Question:** MENTORX predicts whether a student will answer the next maths question correctly with
**DKT-Q**, a one-layer LSTM over the student's answer history. Does a variation of the recurrent part
predict better?

| Model | What changes compared with DKT-Q |
|---|---|
| **DKT-Q (LSTM)**, deployed | nothing; this is the current model |
| **DKT-Q-GRU** | the LSTM cell is replaced by a GRU: 2 gates instead of 3, no separate cell state, fewer parameters |
| **DKT-Q-LSTM×2** | two stacked LSTM layers (dropout between them) instead of one |
| **DKT-Q-LSTM+Attn** | the LSTM, plus attention from the *next question* to all earlier hidden states, so it can look straight back at related old answers |

Everything else is identical for all four: inputs (question, skills, correct/incorrect, hints, answer
shown, time gap of earlier attempts), the prediction head (question embedding + per-question difficulty),
optimiser, early stopping, data cleaning and the **student split (70/15/15, seed 42)** used to train the
deployed model (notebook 02).

**Fair evaluation:**
- Each model is trained with **3 training seeds** (the data split never changes), so we can see how much
  of a difference is just training noise.
- The variant to present is chosen on **validation** students only. The **test** students are scored once.
- 95% confidence intervals resample **students**. The paired interval for "variant − DKT-Q" uses the same
  resampled students for both models.
- Besides accuracy, it compares calibration (MENTORX picks questions at P(correct) ≈ 0.65), model size,
  training time and CPU inference time (the server has no GPU).

**How to run on Kaggle**
1. *Create → New Notebook → File → Import Notebook*, and upload this file.
2. *Add Input* → attach the **foundational-assist-dataset** (`Interactions.csv`, `Problems.csv`, `Skills.csv`).
3. *Settings → Accelerator → GPU T4 x1*.
4. *Save Version → Save & Run All (Commit)*. You can close the browser.
5. When it finishes: version → *Output* → download the `mentorx_lstm` folder, and send back what the
   last cell prints.

In [ ]:
import copy
import gc
import json
import math
import os
import platform
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import accuracy_score, brier_score_loss, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split

SPLIT_SEED = 42  # the data split; never changes
SMOKE = os.environ.get("MENTORX_SMOKE") == "1"  # tiny local test only; never set on Kaggle
TRAIN_SEEDS = [42] if SMOKE else [42, 7, 123]


def set_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SPLIT_SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_DIR = Path(os.environ.get("MENTORX_DATA_DIR", "/kaggle/input"))
if not (DATA_DIR / "Interactions.csv").exists():
    found = sorted(DATA_DIR.rglob("Interactions.csv")) if DATA_DIR.exists() else []
    if not found:
        raise FileNotFoundError("Attach the foundational-assist-dataset to this notebook (Add Input).")
    DATA_DIR = found[0].parent
OUT_DIR = Path(os.environ.get("MENTORX_OUT_DIR", "/kaggle/working/mentorx_lstm"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Same training settings as the deployed DKT-Q (notebook 02)
MAX_EPOCHS = 1 if SMOKE else 30
PATIENCE = 3
BATCH_SIZE = 64
LR = 1e-3
WEIGHT_DECAY = 1e-5
DIM = 128
DROPOUT = 0.2

MODELS = {  # display name -> recurrent part
    "DKT-Q (LSTM)": "lstm",
    "DKT-Q-GRU": "gru",
    "DKT-Q-LSTM×2": "lstm2",
    "DKT-Q-LSTM+Attn": "lstm_attn",
}
BASELINE = "DKT-Q (LSTM)"

if DEVICE.type == "cpu" and not SMOKE:
    print("WARNING: no GPU. Settings -> Accelerator -> GPU T4.")
print("Data:", DATA_DIR, "| Outputs:", OUT_DIR.resolve())
print("Device:", DEVICE, torch.cuda.get_device_name(0) if torch.cuda.is_available() else "",
      "| torch", torch.__version__, "| seeds", TRAIN_SEEDS, "| smoke" if SMOKE else "")

## 1. Load and clean (identical to notebook 02)

In [ ]:
raw = pd.read_csv(DATA_DIR / "Interactions.csv", index_col=0)
problems_raw = pd.read_csv(DATA_DIR / "Problems.csv")
skills_raw = pd.read_csv(DATA_DIR / "Skills.csv")

df = raw.drop_duplicates(subset="id").dropna(subset=["discrete_score"]).copy()
df["end_time"] = pd.to_datetime(df["end_time"], utc=True, format="ISO8601", errors="coerce")
df = df.dropna(subset=["end_time"])
df["correct"] = df["discrete_score"].astype(np.int8)
df = df.sort_values(["user_id", "end_time", "id"]).reset_index(drop=True)
skills = skills_raw.drop_duplicates(subset=["problem_id", "skill_id"])
print(f"Interactions: {len(raw):,} raw -> {len(df):,} clean")
del raw
df = df.drop(columns=["answer_text", "discrete_score"])
gc.collect()

if SMOKE:  # 120 students so a CPU test finishes in minutes
    keep = np.random.default_rng(0).choice(df.user_id.unique(), 120, replace=False)
    df = df[df.user_id.isin(keep)].reset_index(drop=True)

print(f"Students {df.user_id.nunique():,} | questions {df.problem_id.nunique():,} | "
      f"skills {skills.skill_id.nunique()} | correct rate {df.correct.mean():.3f}")

## 2. Index skills and questions, split by student (identical to notebook 02)

A question that no training student attempted maps to index 0 ("unknown").

In [ ]:
student_ids = np.sort(df["user_id"].unique())
train_students, temp = train_test_split(student_ids, test_size=0.30, random_state=SPLIT_SEED)
val_students, test_students = train_test_split(temp, test_size=0.50, random_state=SPLIT_SEED)
split_of = pd.Series("train", index=student_ids)
split_of[val_students] = "val"
split_of[test_students] = "test"
df["split"] = df["user_id"].map(split_of).values

skill_ids = np.sort(skills["skill_id"].unique())
skill_idx = {s: i + 1 for i, s in enumerate(skill_ids)}
NUM_SKILLS = len(skill_ids) + 1
MAX_SK = 3
prob_skills = skills.groupby("problem_id")["skill_id"].apply(lambda s: [skill_idx[x] for x in s][:MAX_SK])
sk = np.zeros((len(df), MAX_SK), dtype=np.int64)
for j, lst in enumerate(df["problem_id"].map(prob_skills)):
    if isinstance(lst, list):
        sk[j, : len(lst)] = lst

train_q = np.sort(df.loc[df.split == "train", "problem_id"].unique())
q_idx = {q: i + 1 for i, q in enumerate(train_q)}
NUM_Q = len(train_q) + 1
df["q"] = df["problem_id"].map(q_idx).fillna(0).astype(np.int64)

for name in ["train", "val", "test"]:
    part = df[df.split == name]
    print(f"{name:5s}: {part.user_id.nunique():5,} students  {len(part):9,} interactions  "
          f"unknown-question rows {(part.q == 0).mean():.2%}")

## 3. Metrics, first-attempt fallback, sequences (identical to notebook 02)

In [ ]:
def ece_score(y, p, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges) - 1, 0, bins - 1)
    return sum((idx == b).mean() * abs(y[idx == b].mean() - p[idx == b].mean()) for b in range(bins) if (idx == b).any())


def metrics(y, p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    m = {"AUC": roc_auc_score(y, p), "LogLoss": log_loss(y, p, labels=[0, 1]),
         "Accuracy": accuracy_score(y, p >= 0.5), "Brier": brier_score_loss(y, p), "ECE": ece_score(y, p)}
    return {k: float(v) for k, v in m.items()}


val_rows = df.index[df.split == "val"].to_numpy()
test_rows = df.index[df.split == "test"].to_numpy()
y_val = df.loc[val_rows, "correct"].to_numpy()
y_test = df.loc[test_rows, "correct"].to_numpy()

tr = df[df.split == "train"]
glob = tr.correct.mean()
g = tr.groupby("problem_id")["correct"].agg(["sum", "count"])
q_rate = (g["sum"] + 2 * glob) / (g["count"] + 2)


def first_attempt_fill(p, rows):
    # A student's very first attempt has no history; every model falls back to the question's success rate
    out = p[rows].copy()
    miss = np.isnan(out)
    out[miss] = df.loc[rows[miss], "problem_id"].map(q_rate).fillna(glob).to_numpy()
    return out


secs_all = df["end_time"].astype("int64").to_numpy() / 1e9
users = df["user_id"].to_numpy()
new_user = np.r_[True, users[1:] != users[:-1]]
gap = np.r_[0.0, np.diff(secs_all)]
gap[new_user] = 0.0
feat_all = np.stack([
    np.log1p(df["hint_count"].clip(upper=10).to_numpy()),
    df["saw_answer"].astype(float).to_numpy(),
    np.log1p(np.maximum(gap, 0)) / 10.0,
], axis=1).astype(np.float32)

USER_START = np.flatnonzero(new_user)
USER_END = np.r_[USER_START[1:], len(df)]
Q_ALL = df["q"].to_numpy().copy()
R_ALL = df["correct"].to_numpy().astype(np.int64)
SPLIT_ALL = df["split"].to_numpy()
SEQ = {s: [] for s in ["train", "val", "test"]}
for a, b in zip(USER_START, USER_END):
    if b - a >= 2:
        SEQ[SPLIT_ALL[a]].append({"rows": np.arange(a, b), "q": Q_ALL[a:b], "sk": sk[a:b], "r": R_ALL[a:b], "f": feat_all[a:b]})
lens = np.array([len(x["r"]) for v in SEQ.values() for x in v])
print({k: len(v) for k, v in SEQ.items()}, "| attempts per student: median", int(np.median(lens)), "max", int(lens.max()))


def collate(batch):
    # Position t holds attempt t (input) and attempt t+1 (the one to predict)
    L = max(len(x["r"]) for x in batch) - 1
    B = len(batch)
    out = {"q_in": torch.zeros(B, L, dtype=torch.long), "sk_in": torch.zeros(B, L, MAX_SK, dtype=torch.long),
           "r_in": torch.zeros(B, L, dtype=torch.long), "f_in": torch.zeros(B, L, 3),
           "q_out": torch.zeros(B, L, dtype=torch.long), "sk_out": torch.zeros(B, L, MAX_SK, dtype=torch.long),
           "y": torch.zeros(B, L), "mask": torch.zeros(B, L, dtype=torch.bool), "lengths": torch.zeros(B, dtype=torch.long)}
    for i, x in enumerate(batch):
        n = len(x["r"]) - 1
        out["q_in"][i, :n] = torch.from_numpy(x["q"][:-1])
        out["sk_in"][i, :n] = torch.from_numpy(x["sk"][:-1])
        out["r_in"][i, :n] = torch.from_numpy(x["r"][:-1])
        out["f_in"][i, :n] = torch.from_numpy(x["f"][:-1])
        out["q_out"][i, :n] = torch.from_numpy(x["q"][1:])
        out["sk_out"][i, :n] = torch.from_numpy(x["sk"][1:])
        out["y"][i, :n] = torch.from_numpy(x["r"][1:].astype(np.float32))
        out["mask"][i, :n] = True
        out["lengths"][i] = n
    return out


def to_device(b):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}

## 4. The four models

One class, four settings. Only the recurrent encoder (and, for the attention variant, one extra block)
changes:

- **LSTM** (deployed): input, forget and output gates plus a separate cell state.
- **GRU**: update and reset gates, and the hidden state doubles as the memory. About 25% fewer recurrent
  weights than the LSTM.
- **LSTM×2**: a second LSTM layer reads the first layer's outputs, giving a deeper summary of the history.
- **LSTM+Attn**: multi-head attention where the query is the *next question* (its skills and question
  embedding) and the keys/values are the LSTM states of all **earlier** attempts. An LSTM squeezes the
  whole history into one vector; attention can fetch, for example, the answer to a similar question from
  200 attempts ago.

A bidirectional LSTM is *not* an option: it would read future answers, which a tutor cannot know.
The leakage test below checks that no variant can see the answer it is predicting.

In [ ]:
class SkillBag(nn.Module):
    # Mean of up to 3 skill embeddings (index 0 = padding)
    def __init__(self, n, dim):
        super().__init__()
        self.emb = nn.Embedding(n, dim, padding_idx=0)

    def forward(self, idx):
        return self.emb(idx).sum(-2) / (idx > 0).sum(-1, keepdim=True).clamp(min=1)


class KTVariant(nn.Module):
    def __init__(self, kind="lstm", dim=DIM, dropout=DROPOUT, heads=4):
        super().__init__()
        self.kind = kind
        self.skill_resp = SkillBag(2 * NUM_SKILLS, dim)
        self.skill_out = SkillBag(NUM_SKILLS, dim)
        self.q_resp = nn.Embedding(2 * NUM_Q, dim, padding_idx=0)
        self.q_out = nn.Embedding(NUM_Q, dim, padding_idx=0)
        self.q_bias = nn.Embedding(NUM_Q, 1, padding_idx=0)  # per-question difficulty
        nn.init.zeros_(self.q_bias.weight)
        self.inp = nn.Sequential(nn.Linear(2 * dim + 3, dim), nn.LayerNorm(dim), nn.Dropout(dropout))
        if kind == "gru":
            self.enc = nn.GRU(dim, dim, batch_first=True)
        elif kind == "lstm2":
            self.enc = nn.LSTM(dim, dim, num_layers=2, dropout=dropout, batch_first=True)
        else:
            self.enc = nn.LSTM(dim, dim, batch_first=True)
        head_in = 3 * dim
        if kind == "lstm_attn":
            self.query = nn.Linear(2 * dim, dim)
            self.attn = nn.MultiheadAttention(dim, heads, dropout=dropout, batch_first=True)
            self.attn_norm = nn.LayerNorm(dim)
            head_in = 4 * dim
        self.head = nn.Sequential(nn.Linear(head_in, dim), nn.ReLU(), nn.Dropout(dropout), nn.Linear(dim, 1))

    def forward(self, b):
        sk_in, r_in = b["sk_in"], b["r_in"]
        sr = torch.where(sk_in > 0, 2 * sk_in + r_in.unsqueeze(-1), torch.zeros_like(sk_in))
        qr = torch.where(b["q_in"] > 0, 2 * b["q_in"] + r_in, torch.zeros_like(r_in))
        x = self.inp(torch.cat([self.skill_resp(sr), self.q_resp(qr), b["f_in"]], -1))
        L = x.size(1)
        packed = nn.utils.rnn.pack_padded_sequence(x, b["lengths"].cpu(), batch_first=True, enforce_sorted=False)
        h, _ = self.enc(packed)
        h, _ = nn.utils.rnn.pad_packed_sequence(h, batch_first=True, total_length=L)
        target = [self.skill_out(b["sk_out"]), self.q_out(b["q_out"])]  # the question to predict (no outcome)
        parts = [h] + target
        if self.kind == "lstm_attn":
            # position t may look at states 0..t (attempts 0..t), all earlier than attempt t+1
            future = torch.triu(torch.ones(L, L, dtype=torch.bool, device=x.device), 1)
            ctx, _ = self.attn(self.query(torch.cat(target, -1)), h, h, attn_mask=future, need_weights=False)
            parts.append(self.attn_norm(ctx))
        logit = self.head(torch.cat(parts, -1)).squeeze(-1)
        return logit + self.q_bias(b["q_out"]).squeeze(-1)


# Leakage test: changing attempt 7 (its answer, hints, time) must not change predictions of attempts 1..7
rng = np.random.default_rng(0)
fake = {"rows": np.arange(13), "q": rng.integers(1, NUM_Q, 13), "sk": rng.integers(1, NUM_SKILLS, (13, MAX_SK)),
        "r": rng.integers(0, 2, 13), "f": rng.random((13, 3)).astype(np.float32)}
fake2 = {**fake, "r": fake["r"].copy(), "f": fake["f"].copy()}
fake2["r"][7] = 1 - fake2["r"][7]
fake2["f"][7] += 1
for name, kind in MODELS.items():
    m = KTVariant(kind).eval()
    with torch.no_grad():
        p1, p2 = m(collate([fake]))[0], m(collate([fake2]))[0]
    assert torch.allclose(p1[:7], p2[:7], atol=1e-6), f"{name} sees the answer it predicts"
    assert not torch.allclose(p1[7:], p2[7:]), f"{name} ignores earlier answers"
    print(f"{name:18s} leakage test passed | {sum(p.numel() for p in m.parameters()):,} parameters")

## 5. Training: 4 models × 3 seeds

Same loop as the deployed model: AdamW (lr 1e-3), learning rate halved on a validation plateau, early
stopping on validation log loss (patience 3), gradient clipping at 1.0, batch 64. The best epoch of each
run is kept.

In [ ]:
def ckpt_path(name, seed):
    slug = name.lower().replace("×", "x").replace("+", "_").replace(" ", "_").replace("(", "").replace(")", "").replace("-", "_")
    return OUT_DIR / f"{slug}_seed{seed}.pt"


@torch.no_grad()
def predict(model, split):
    model.eval()
    probs = np.full(len(df), np.nan)
    seqs = SEQ[split]
    for i in range(0, len(seqs), BATCH_SIZE):
        chunk = seqs[i:i + BATCH_SIZE]
        p = torch.sigmoid(model(to_device(collate(chunk)))).cpu().numpy()
        for j, x in enumerate(chunk):
            probs[x["rows"][1:]] = p[j, : len(x["r"]) - 1]
    return probs


def train_one(name, kind, seed):
    path = ckpt_path(name, seed)
    set_seed(seed)
    model = KTVariant(kind).to(DEVICE)
    if path.exists() and os.environ.get("RETRAIN") != "1":
        ck = torch.load(path, map_location=DEVICE)
        model.load_state_dict(ck["state_dict"])
        print(f"{name} seed {seed}: loaded {path.name}")
        return model, pd.DataFrame(ck["history"])
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=1)
    gen = torch.Generator().manual_seed(seed)
    dl = torch.utils.data.DataLoader(SEQ["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate, generator=gen)
    vr = np.flatnonzero(SPLIT_ALL == "val")
    best, bad, hist, best_state = math.inf, 0, [], None
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0, tot, cnt = time.time(), 0.0, 0
        for b in dl:
            b = to_device(b)
            m = b["mask"]
            loss_sum = F.binary_cross_entropy_with_logits(model(b)[m], b["y"][m], reduction="sum")
            opt.zero_grad(set_to_none=True)
            (loss_sum / m.sum()).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            tot += loss_sum.item()
            cnt += m.sum().item()
        if DEVICE.type == "cuda":
            torch.cuda.synchronize()
        secs = time.time() - t0
        p = predict(model, "val")[vr]
        keep = ~np.isnan(p)
        mv = metrics(R_ALL[vr][keep], p[keep])
        sched.step(mv["LogLoss"])
        hist.append({"epoch": epoch, "train_loss": tot / cnt, **{f"val_{k}": v for k, v in mv.items()}, "train_seconds": secs})
        line = (f"{name} seed {seed} epoch {epoch:2d} | train {tot / cnt:.4f} | val logloss {mv['LogLoss']:.4f} "
                f"AUC {mv['AUC']:.4f} | lr {opt.param_groups[0]['lr']:.0e} | {secs:.1f}s")
        print(line, flush=True)
        with open(OUT_DIR / "train_log.txt", "a") as fh:
            fh.write(line + "\n")
        if mv["LogLoss"] < best - 1e-4:
            best, bad, best_state = mv["LogLoss"], 0, copy.deepcopy(model.state_dict())
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    torch.save({"state_dict": best_state, "kind": kind, "seed": seed, "num_skills": NUM_SKILLS, "num_q": NUM_Q,
                "dim": DIM, "history": hist}, path)
    return model, pd.DataFrame(hist)


RUNS = {}  # (name, seed) -> {"val": probs, "test": probs, "history": df, "params": n}
for name, kind in MODELS.items():
    for seed in TRAIN_SEEDS:
        model, hist = train_one(name, kind, seed)
        RUNS[(name, seed)] = {
            "val": first_attempt_fill(predict(model, "val"), val_rows),
            "test": first_attempt_fill(predict(model, "test"), test_rows),  # stored now, looked at only in §7
            "history": hist,
            "params": sum(p.numel() for p in model.parameters()),
        }
        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

## 6. Validation results and the choice of variant

The variant to present is the one with the lowest **mean validation log loss over the 3 seeds**.
Log loss measures probability quality, which is what MENTORX acts on.

In [ ]:
def summary(split, y):
    rows = []
    for name in MODELS:
        ms = [metrics(y, RUNS[(name, s)][split]) for s in TRAIN_SEEDS]
        row = {"Model": name}
        for k in ["AUC", "LogLoss", "Accuracy", "Brier", "ECE"]:
            vals = np.array([m[k] for m in ms])
            row[k] = vals.mean()
            row[f"{k} sd"] = vals.std(ddof=1) if len(vals) > 1 else 0.0
        rows.append(row)
    return pd.DataFrame(rows).set_index("Model")


val_table = summary("val", y_val)
display(val_table.round(4))
variants = [m for m in MODELS if m != BASELINE]
CHOSEN = min(variants, key=lambda m: val_table.loc[m, "LogLoss"])
print(f"Chosen LSTM variation (validation log loss): {CHOSEN}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name in MODELS:
    h = RUNS[(name, TRAIN_SEEDS[0])]["history"]
    axes[0].plot(h["epoch"], h["val_AUC"], marker="o", label=name)
    axes[1].plot(h["epoch"], h["val_LogLoss"], marker="o", label=name)
axes[0].set(title=f"Validation AUC (seed {TRAIN_SEEDS[0]})", xlabel="epoch")
axes[1].set(title=f"Validation log loss (seed {TRAIN_SEEDS[0]})", xlabel="epoch")
axes[0].legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "learning_curves.png", dpi=150)
plt.show()

## 7. Test students (scored once)

- **Main table:** mean ± sd over the 3 training seeds.
- **Confidence intervals and the paired difference** use the seed-42 run of each model. If the paired
  interval for "variant − DKT-Q" lies entirely above 0, the variant is better. If it contains 0, they
  are tied.

In [ ]:
test_table = summary("test", y_test)

S0 = TRAIN_SEEDS[0]
groups = df.loc[test_rows, "user_id"].to_numpy()
uniq, inv = np.unique(groups, return_inverse=True)
rows_of = [np.flatnonzero(inv == k) for k in range(len(uniq))]
brng = np.random.default_rng(SPLIT_SEED)
REPS = 50 if SMOKE else 500
boot = {m: [] for m in MODELS}
for _ in range(REPS):
    pick = np.concatenate([rows_of[k] for k in brng.integers(0, len(uniq), len(uniq))])
    for m in MODELS:
        boot[m].append(roc_auc_score(y_test[pick], RUNS[(m, S0)]["test"][pick]))
boot = {m: np.array(v) for m, v in boot.items()}

ci, paired = {}, {}
for m in MODELS:
    lo, hi = np.percentile(boot[m], [2.5, 97.5])
    ci[m] = (float(lo), float(hi))
    if m != BASELINE:
        point = metrics(y_test, RUNS[(m, S0)]["test"])["AUC"] - metrics(y_test, RUNS[(BASELINE, S0)]["test"])["AUC"]
        dlo, dhi = np.percentile(boot[m] - boot[BASELINE], [2.5, 97.5])
        verdict = "better than DKT-Q" if dlo > 0 else ("worse than DKT-Q" if dhi < 0 else "tied with DKT-Q")
        paired[m] = {"diff": float(point), "ci": [float(dlo), float(dhi)], "verdict": verdict}
test_table[f"AUC 95% CI (seed {S0})"] = [f"{ci[m][0]:.3f}-{ci[m][1]:.3f}" for m in test_table.index]
display(test_table.round(4))
for m, d in paired.items():
    print(f"AUC({m}) - AUC(DKT-Q) = {d['diff']:+.4f}  95% CI {d['ci'][0]:+.4f} to {d['ci'][1]:+.4f}  -> {d['verdict']}")

## 8. Charts for the presentation

In [ ]:
names = list(MODELS)
colors = {n: c for n, c in zip(names, ["#3b4a8f", "#7a86d6", "#9aa3b5", "#c58b3a"])}

# AUC with 95% CI (seed 42) and seed spread
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(names))
auc0 = [metrics(y_test, RUNS[(n, S0)]["test"])["AUC"] for n in names]
err = np.array([[auc0[i] - ci[n][0], ci[n][1] - auc0[i]] for i, n in enumerate(names)]).T
ax.bar(x, auc0, color=[colors[n] for n in names], yerr=err, capsize=6)
for i, n in enumerate(names):
    for s in TRAIN_SEEDS:
        ax.plot(i, metrics(y_test, RUNS[(n, s)]["test"])["AUC"], "k.", ms=6)
lo = min(c[0] for c in ci.values()) - 0.01
ax.set(ylim=(lo, max(c[1] for c in ci.values()) + 0.005), xticks=x, ylabel="Test AUC",
       title="Test AUC (bars: seed 42 with 95% CI; dots: each seed)")
ax.set_xticklabels(names, rotation=10)
plt.tight_layout()
plt.savefig(OUT_DIR / "auc_comparison.png", dpi=150)
plt.show()

# Reliability: baseline vs chosen variant
fig, ax = plt.subplots(figsize=(5, 5))
for n in [BASELINE, CHOSEN]:
    p = RUNS[(n, S0)]["test"]
    bins = np.clip((p * 10).astype(int), 0, 9)
    ok = [b for b in range(10) if (bins == b).sum() > 100]
    ax.plot([p[bins == b].mean() for b in ok], [y_test[bins == b].mean() for b in ok], marker="o", label=n, color=colors[n])
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set(xlabel="predicted P(correct)", ylabel="observed correct rate", title="Calibration (test students)")
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "calibration.png", dpi=150)
plt.show()

# MENTORX targets P(correct) = 0.65: is the band 0.55-0.75 honest?
BAND = {}
for n in names:
    p = RUNS[(n, S0)]["test"]
    band = (p >= 0.55) & (p <= 0.75)
    BAND[n] = {"n": int(band.sum()), "predicted": float(p[band].mean()), "observed": float(y_test[band].mean())}
    print(f"{n:18s} band 0.55-0.75: n={band.sum():,} predicted={p[band].mean():.3f} observed={y_test[band].mean():.3f}")

## 9. Cost: size, training time, CPU inference time

MENTORX serves on a CPU-only server, so inference is timed on the CPU: one student with 300 earlier
attempts (a typical history here), median of 30 runs, single thread.

In [ ]:
torch.set_num_threads(1)
x300 = SEQ["test"][0] if len(SEQ["test"][0]["r"]) >= 301 else max(SEQ["test"], key=lambda s: len(s["r"]))
one = {k: (v[:301] if k != "rows" else v[:301]) for k, v in x300.items()}
b1 = collate([one])
COST = {}
for n, kind in MODELS.items():
    m = KTVariant(kind)
    m.load_state_dict(torch.load(ckpt_path(n, S0), map_location="cpu")["state_dict"])
    m.eval()
    with torch.no_grad():
        for _ in range(3):
            m(b1)
        ts = []
        for _ in range(30):
            t0 = time.perf_counter()
            m(b1)
            ts.append(time.perf_counter() - t0)
    hists = [RUNS[(n, s)]["history"] for s in TRAIN_SEEDS]
    COST[n] = {
        "parameters": RUNS[(n, S0)]["params"],
        "recurrent_parameters": sum(p.numel() for p in m.enc.parameters()) + (
            sum(p.numel() for p in m.attn.parameters()) if kind == "lstm_attn" else 0),
        "epochs_to_best (mean)": float(np.mean([h["val_LogLoss"].idxmin() + 1 for h in hists])),
        "seconds_per_epoch (GPU)": float(np.mean([h["train_seconds"].mean() for h in hists])),
        "cpu_ms_per_student": 1000 * float(np.median(ts)),
        "history_length": len(one["r"]) - 1,
    }
torch.set_num_threads(os.cpu_count() or 4)
cost_table = pd.DataFrame(COST).T
display(cost_table.round(2))

## 10. Save and summary

`mentorx_lstm/` holds every checkpoint (`<model>_seed<n>.pt`), `results_lstm.json`, `train_log.txt`
and the charts (`auc_comparison.png`, `calibration.png`, `learning_curves.png`).

In [ ]:
results = {
    "dataset": "FoundationalASSIST (ASSISTments; CC BY-NC 4.0)",
    "split": "by student 70/15/15, seed 42 (same as notebook 02 and the deployed DKT-Q)",
    "smoke_test": SMOKE, "train_seeds": TRAIN_SEEDS, "baseline": BASELINE, "chosen_variant": CHOSEN,
    "settings": {"max_epochs": MAX_EPOCHS, "patience": PATIENCE, "batch": BATCH_SIZE, "lr": LR, "dim": DIM, "dropout": DROPOUT},
    "val_mean_over_seeds": val_table.round(5).to_dict(orient="index"),
    "test_mean_over_seeds": test_table.round(5).to_dict(orient="index"),
    "test_per_seed": {f"{n} | seed {s}": metrics(y_test, RUNS[(n, s)]["test"]) for n in MODELS for s in TRAIN_SEEDS},
    "test_auc_ci_seed42": ci, "paired_vs_baseline": paired, "band_055_075": BAND, "cost": COST,
    "environment": {"device": str(DEVICE), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
                    "torch": torch.__version__, "python": platform.python_version()},
}
json.dump(results, open(OUT_DIR / "results_lstm.json", "w"), indent=2, default=float)

pd.set_option("display.width", 200)
print("=" * 30, "MENTORX LSTM VARIANTS SUMMARY", "=" * 30)
print(f"device={DEVICE} gpu={results['environment']['gpu']} torch={torch.__version__} rows={len(df):,} "
      f"students={df.user_id.nunique():,} seeds={TRAIN_SEEDS} smoke={SMOKE}")
print(f"chosen variant (validation): {CHOSEN}")
print("\n--- validation (mean over seeds) ---")
print(val_table.round(4).to_string())
print("\n--- test (mean over seeds) ---")
print(test_table.round(4).to_string())
print()
for m, d in paired.items():
    print(f"AUC({m}) - AUC(DKT-Q) = {d['diff']:+.4f}  CI95 {d['ci'][0]:+.4f} to {d['ci'][1]:+.4f}  -> {d['verdict']}")
print("\n--- cost ---")
print(cost_table.round(2).to_string())
print("\nFiles:", sorted(p.name for p in OUT_DIR.iterdir()))
print("=" * 91)